# AMEX Enterprise Credit Risk Platform
## Notebook 01 — Business Understanding
### Phase 1 · Problem Statement 1: Credit Scoring / PD Prediction

CRISP-DM stage: **Business Understanding**. Sprint 1, Notebook 1 of 18. This notebook takes no dependency on any other notebook — it is the entry point for the whole build.

**Zero-fabrication rule, applied throughout this platform:** every number shown anywhere in this notebook, its generated Word documents, or the final consolidated reports is either (a) computed live by that section's own code from the real files on this machine, or (b) explicitly marked `PENDING` because the notebook that computes it has not been run yet. Nothing is carried over from any prior session, and nothing is hand-typed as if it were a result. Model performance metrics in particular do not exist until Notebook 05 trains a model — this notebook (01) never states one. Notebook 17 later builds the consolidated Word/Excel/HTML/dashboard reports purely by auto-reading the JSON/CSV artifact each notebook writes to `artifacts/` — never by retyping values.

**Run the single code cell below, once.** All 11 sections are consolidated into that one cell — each still prints its own clearly labeled banner as it runs, in order, and every deliverable is verified to exist on disk before the cell reports itself complete. Re-running the cell is safe: every output file below is written to the same fixed path each time and overwritten in place -- re-running never creates a second, third, or renamed copy of anything.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP & CONFIGURATION
# =============================================================================
import os
import sys
import json
import logging
import platform
import multiprocessing
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    """Print a clearly labeled banner before each section's real work runs.
    Defined once, here, and reused by every notebook in this build (WARP
    Section 6.4 — no repeated formatting logic scattered across 18 notebooks)."""
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup & Configuration")

# --- Project paths (single source of truth). Notebooks 02-18 read PROJECT_ROOT
#     back out of artifacts/project_config.json (written at the end of Section 3)
#     rather than re-hardcoding this path 18 separate times. Edit this one line
#     if you extracted the project folder somewhere else. ---
PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
DATA_ROOT = PROJECT_ROOT.parent / "Raw Data From Kaggle"  # holds train_data.csv, test_data.csv, train_labels.csv, sample_submission.csv
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

PILLAR_DIRS = {
    "business_documents": PROJECT_ROOT / "Business_Documents",
    "data_engineering": PROJECT_ROOT / "Data_Engineering",
    "data_validation": PROJECT_ROOT / "Data_Validation",
    "feature_engineering": PROJECT_ROOT / "Feature_Engineering",
    "model_development": PROJECT_ROOT / "Model_Development",
    "explainable_ai": PROJECT_ROOT / "Explainable_AI_SHAP_LIME",
    "model_risk_management": PROJECT_ROOT / "Model_Risk_Management",
    "basel_ifrs9": PROJECT_ROOT / "Basel_III_IFRS9_Mapping",
    "mlops": PROJECT_ROOT / "MLOps",
    "fastapi_deployment": PROJECT_ROOT / "FastAPI_Deployment",
    "docker": PROJECT_ROOT / "Docker",
    "monitoring": PROJECT_ROOT / "Monitoring",
    "powerbi_dashboard": PROJECT_ROOT / "PowerBI_Dashboard",
    "executive_reports": PROJECT_ROOT / "Executive_Reports",
    "technical_documentation": PROJECT_ROOT / "Technical_Documentation",
    "production_architecture": PROJECT_ROOT / "Production_Architecture",
    "comprehensive_reporting": PROJECT_ROOT / "Comprehensive_Reporting",
    "repository_packaging": PROJECT_ROOT / "Repository_Packaging",
}

# Idempotent: safe to re-run this cell any number of times without side effects.
for _path in [ARTIFACTS_DIR, NOTEBOOKS_DIR, *PILLAR_DIRS.values()]:
    _path.mkdir(parents=True, exist_ok=True)

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"DATA_ROOT does not exist: {DATA_ROOT}\n"
        "Fix: confirm the official AMEX CSVs (train_data.csv, test_data.csv, train_labels.csv, "
        "sample_submission.csv) live in a 'Raw Data From Kaggle' subfolder directly under the "
        "folder containing AMEX_Enterprise_Credit_Risk_Platform -- or edit PROJECT_ROOT/DATA_ROOT "
        "above to match wherever they actually live on this machine, then re-run this cell."
    )

print(f"PROJECT_ROOT          : {PROJECT_ROOT}")
print(f"DATA_ROOT              : {DATA_ROOT}")
print(f"ARTIFACTS_DIR          : {ARTIFACTS_DIR}")
print(f"Pillar folders ready   : {len(PILLAR_DIRS)}")
print("Status: all project directories exist and are ready. \u2705")


# =============================================================================
# SECTION 2: LOGGING CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: Logging Configuration & Library Imports")

# Structured logging instead of scattered print() calls (Master Execution Plan,
# Section 12: Quality & Reliability Standards). StreamHandler targets stdout so
# every log line still renders inline in Jupyter exactly like print() would.
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

logger.info("Logger initialized for notebook 01_business_understanding")

# --- Required libraries for THIS notebook only. Business Understanding is
#     deliberately dependency-light: it produces two Word documents and one
#     CSV, so it needs python-docx plus the standard library — not
#     Polars/LightGBM/etc., which notebooks 02+ import when they actually
#     touch the raw AMEX data. Importing only what a notebook uses keeps each
#     stage fast and stops a missing dependency in one area from hiding
#     behind an unrelated notebook's success. ---
missing = []
try:
    import docx  # python-docx
    from docx import Document
except ImportError:
    missing.append("python-docx")

import csv  # stdlib — Stakeholder_Analysis.csv is a handful of short rows;
            # no need for pandas/Polars at this stage.

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )

logger.info("All required libraries imported successfully: python-docx, csv (stdlib)")
print("\u2705 Imports complete.")


# =============================================================================
# SECTION 3: REPRODUCIBILITY & HARDWARE CONFIGURATION
# =============================================================================
_section("SECTION 3: Reproducibility & Hardware Configuration")

RANDOM_SEED = 42  # reused by every downstream notebook (05 model training,
                   # train/val splits, etc.) for reproducible results

logical_cores = multiprocessing.cpu_count()
try:
    import psutil
    physical_cores = psutil.cpu_count(logical=False)
    total_ram_bytes = psutil.virtual_memory().total
    total_ram_gb = round(total_ram_bytes / (1024 ** 3), 1)
except ImportError:
    physical_cores = None
    total_ram_bytes = None
    total_ram_gb = None
    logger.warning(
        "psutil not installed -- physical core count / RAM size will show as null in "
        "project_config.json. Optional: pip install psutil for the full hardware readout."
    )

hardware_info = {
    "platform": platform.platform(),
    "python_version": platform.python_version(),
    "logical_cores_detected": logical_cores,
    "physical_cores_detected": physical_cores,
    "total_ram_gb_detected": total_ram_gb,
}

# --- Resource ceilings: cap every downstream notebook to a configured share of
#     this machine's threads and RAM, rather than assuming it can have all of
#     it. Two honest caveats up front:
#       1. CPU CLOCK SPEED (e.g. "95% of 5GHz") is NOT something a Python
#          process can set -- boost clocks are managed by the OS power plan
#          and the CPU's own firmware, not application code. Nothing below
#          claims to control clock speed.
#       2. What IS enforced is a THREAD-COUNT ceiling (95% of logical cores,
#          rounded down) applied everywhere a library takes an explicit
#          thread/process count (Polars, scikit-learn, XGBoost, LightGBM,
#          CatBoost), and a RAM ceiling (90% of detected total RAM) that
#          downstream notebooks use to choose memory-efficient dtypes
#          (float32 over float64), free large intermediate objects promptly,
#          and process very large arrays in bounded chunks instead of loading
#          everything into memory as one block. ---
RAM_FRACTION_CAP = 0.90
CPU_FRACTION_CAP = 0.95
WARP_THREAD_COUNT = max(1, int(logical_cores * CPU_FRACTION_CAP)) if logical_cores else 1
MAX_RAM_BYTES = int(total_ram_bytes * RAM_FRACTION_CAP) if total_ram_bytes else None
MAX_RAM_GB = round(MAX_RAM_BYTES / (1024 ** 3), 2) if MAX_RAM_BYTES else None

resource_limits = {
    "ram_fraction_cap": RAM_FRACTION_CAP,
    "cpu_fraction_cap": CPU_FRACTION_CAP,
    "total_ram_bytes_detected": total_ram_bytes,
    "max_ram_bytes": MAX_RAM_BYTES,
    "max_ram_gb": MAX_RAM_GB,
    "logical_cores_detected": logical_cores,
    "warp_thread_count": WARP_THREAD_COUNT,
    "clock_speed_note": (
        "CPU clock speed is not application-settable (OS/firmware boost control); "
        "only thread count (warp_thread_count) and a RAM ceiling (max_ram_bytes) are "
        "actually enforced by this platform's notebooks."
    ),
}

logger.info(f"Detected hardware: {hardware_info}")
logger.info(f"Resource ceilings: {resource_limits}")
print(f"Logical cores detected by this notebook : {logical_cores}")
if physical_cores:
    print(f"Physical cores detected                 : {physical_cores}")
if total_ram_gb:
    print(f"Total RAM detected                      : {total_ram_gb} GB")
print(f"WARP_THREAD_COUNT (95% of logical cores) : {WARP_THREAD_COUNT}")
if MAX_RAM_GB:
    print(f"MAX_RAM_GB (90% of detected total RAM)  : {MAX_RAM_GB} GB")
print(
    "\nNote: this reflects whatever machine actually runs this cell. On the target Dell "
    "Ryzen AI 7-350 (8 cores / 16 threads / 32GB), logical_cores_detected should read 16 and "
    "WARP_THREAD_COUNT should read 15 -- Notebook 02 onward reads warp_thread_count and "
    "max_ram_bytes from project_config.json to set Polars/scikit-learn/XGBoost/LightGBM/"
    "CatBoost thread counts and memory-aware chunking explicitly (WARP Section 6.4, "
    "Concurrency), rather than hardcoding these separately in eighteen different notebooks."
)

# --- Persist project-wide configuration once, here, so Notebooks 02-18 read it
#     instead of re-deriving paths / seed / thread-count each time. ---
PROJECT_CONFIG = {
    "project_name": "AMEX Enterprise Credit Risk Platform",
    "config_generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "artifacts_dir": str(ARTIFACTS_DIR),
    "random_seed": RANDOM_SEED,
    "hardware": hardware_info,
    "resource_limits": resource_limits,
    "warp_thread_count": WARP_THREAD_COUNT,
    "pillar_dirs": {k: str(v) for k, v in PILLAR_DIRS.items()},
    "baseline_status": (
        "No model has been trained within this notebook chain yet. Notebook 05 (Model "
        "Development) trains the baseline model and writes its own metrics artifact to "
        "artifacts/ -- that artifact, not any value typed into this or any other notebook, "
        "is what Notebook 17 reads when assembling the final consolidated reports."
    ),
}

config_path = ARTIFACTS_DIR / "project_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(PROJECT_CONFIG, f, indent=2)

logger.info(f"Wrote shared project configuration -> {config_path}")
print(f"\n\u2705 project_config.json written -- every later notebook (02-18) loads this "
      f"file instead of redefining these values.")


# =============================================================================
# SECTION 4: LIVE DATA VERIFICATION & BUSINESS PROBLEM STATEMENT -- PHASE 1, PROBLEM 1
# =============================================================================
_section("SECTION 4: Live Data Verification & Business Problem Statement")

# --- Governing rule for this entire platform: a number appears in the output, a
#     document, or a report ONLY if the owning notebook computed it during that run.
#     Nothing is carried over from any earlier session, and nothing is hand-typed as
#     if it were a result. This cell reads the real, official AMEX files directly
#     (small enough to read fully even in this lightweight notebook) so the
#     population counts and default rate below are live, verifiable facts about
#     the data -- not claims. Model performance is a different kind of fact: it
#     does not exist until a model is trained, so it is correctly marked PENDING
#     everywhere in this notebook and is populated later ONLY from the Notebook 05
#     metrics artifact, read automatically by Notebook 17. ---

labels_path = DATA_ROOT / "train_labels.csv"
sample_sub_path = DATA_ROOT / "sample_submission.csv"
for _p in (labels_path, sample_sub_path):
    if not _p.exists():
        raise FileNotFoundError(
            f"Required file not found: {_p}\n"
            "Fix: confirm DATA_ROOT points at the folder containing the official AMEX CSVs."
        )

_target_sum = 0
_row_count = 0
with open(labels_path, "r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        _target_sum += int(row["target"])
        _row_count += 1

train_customer_count = _row_count
default_rate = _target_sum / _row_count

with open(sample_sub_path, "r", encoding="utf-8") as f:
    test_customer_count = sum(1 for _ in f) - 1  # minus header row

print(f"Live-read {labels_path.name}: {train_customer_count:,} customers, "
      f"{_target_sum:,} defaults -> default rate = {default_rate:.4%}")
print(f"Live-read {sample_sub_path.name}: {test_customer_count:,} customers to score")

BUSINESS_PROBLEM = {
    "phase": "Phase 1 -- Foundation",
    "problem_number": 1,
    "problem_name": "Credit Scoring / Probability of Default (PD) Prediction",
    "depends_on": "None -- this is the foundation every other problem in the 14-problem roadmap consumes",
    "problem_statement": (
        "For every customer in the American Express card portfolio, predict the probability "
        "that the customer defaults (becomes seriously delinquent) within the observation "
        "window used by the AMEX Default Prediction dataset. This single score drives credit "
        "approval decisions, pricing, credit-limit setting, and regulatory capital reservation "
        "at any consumer-credit institution."
    ),
    "why_it_matters": (
        "PD is the single most economically important model at a card issuer: every approval, "
        "every price, every capital reserve traces back to it. Problems 2-14 in this roadmap "
        "either consume this model's output directly (Risk Tier Classification, Expected "
        "Credit Loss) or are variants of the same underlying score computed differently over "
        "time (Behavioral Scoring, Early Warning System)."
    ),
    "success_metric": (
        "Official AMEX competition metric M = 0.5 x Normalized Gini + 0.5 x (default rate "
        "captured in the riskiest 4% of predictions). This is used instead of plain accuracy "
        "or AUC alone because banks specifically need the worst risks ranked correctly at the "
        "top of the list -- not just an accurate model on average."
    ),
    "current_state": (
        f"No model has been trained within this notebook chain yet -- Notebook 01 (Business "
        f"Understanding) does not train or evaluate models, by design. What this cell verifies "
        f"live, by reading the actual files on this machine: {train_customer_count:,} training "
        f"customers in train_labels.csv and {test_customer_count:,} customers to score in "
        f"sample_submission.csv, with a real, computed portfolio default rate of "
        f"{default_rate:.2%}. Notebook 02 builds the customer-level feature store and Notebook "
        f"05 trains and validates the baseline model; its metrics are written to its own "
        f"artifact file and read automatically by Notebook 17 -- never retyped by hand "
        f"anywhere in this platform."
    ),
    "important_dataset_limitation": (
        "The AMEX dataset is an 'accepts-only' dataset: it contains only customers who were "
        "already approved and issued a card. It has no data on rejected applicants. This is "
        "the well-known 'reject inference' problem in credit-risk modeling -- a PD model "
        "trained on this data predicts default risk among ALREADY-APPROVED customers well, "
        "but cannot by itself say how a never-approved applicant population would have "
        "performed. Stated here explicitly so it is not silently assumed away later."
    ),
}

for key, value in BUSINESS_PROBLEM.items():
    label = key.replace("_", " ").title()
    print(f"\n{label}:\n{'-' * len(label)}\n{value}")


# =============================================================================
# SECTION 5: KPI TREE
# =============================================================================
_section("SECTION 5: KPI Tree")

KPI_TREE = [
    {"kpi": "Default Rate", "definition": "Share of customers with target=1 in the booked portfolio",
     "current_value": f"{default_rate:.2%}", "data_source": "Computed live above, this run, from train_labels.csv", "computable_now": True},
    {"kpi": "Validation AUC", "definition": "Area under ROC curve on held-out customers",
     "current_value": "PENDING", "data_source": "Notebook 05 -- auto-populated from its own metrics artifact, never typed by hand", "computable_now": False},
    {"kpi": "AMEX Competition Metric", "definition": "0.5 x Normalized Gini + 0.5 x Top-4% Capture Rate",
     "current_value": "PENDING", "data_source": "Notebook 05 -- auto-populated from its own metrics artifact, never typed by hand", "computable_now": False},
    {"kpi": "Top-4% Default Capture Rate", "definition": "Share of all actual defaulters found in the riskiest 4% of scored customers",
     "current_value": "PENDING", "data_source": "Notebook 05 -- auto-populated from its own metrics artifact, never typed by hand", "computable_now": False},
    {"kpi": "Expected Credit Loss (ECL)", "definition": "PD x LGD x EAD, IFRS9/CECL-style",
     "current_value": "PENDING", "data_source": "Notebook 08 (Basel III / IFRS 9 Mapping)", "computable_now": False},
    {"kpi": "Approval Rate", "definition": "Share of applicants approved for a card",
     "current_value": "NOT COMPUTABLE from this dataset", "data_source": "Requires reject-side application data this dataset does not contain", "computable_now": False},
]

_w = (28, 46, 24, 16)
print(f"{'KPI':<{_w[0]}}{'Definition':<{_w[1]}}{'Current Value':<{_w[2]}}{'Computable Now':<{_w[3]}}")
print("-" * sum(_w))
for row in KPI_TREE:
    print(f"{row['kpi']:<{_w[0]}}{row['definition']:<{_w[1]}}{row['current_value']:<{_w[2]}}{str(row['computable_now']):<{_w[3]}}")

print(
    "\nNote on 'Approval Rate': flagged NOT COMPUTABLE deliberately, rather than silently "
    "omitted or estimated -- see the dataset limitation noted in Section 4."
)


# =============================================================================
# SECTION 6: STAKEHOLDER ANALYSIS
# =============================================================================
_section("SECTION 6: Stakeholder Analysis")

STAKEHOLDERS = [
    {"role": "Chief Risk Officer (CRO)", "interest": "Portfolio-wide default exposure and capital adequacy", "influence": "High", "engagement": "Executive Decision Support Dashboard (Notebook 14)"},
    {"role": "Head of Credit Risk / Underwriting", "interest": "PD model accuracy and approval-policy impact", "influence": "High", "engagement": "Model comparison results, calibration reporting"},
    {"role": "Model Risk Management / Independent Validation", "interest": "SR 11-7-style documentation, bias/fairness testing", "influence": "High", "engagement": "Model Risk Management package (Notebook 07)"},
    {"role": "Compliance / Regulatory Affairs", "interest": "Basel III / IFRS 9 alignment, fair-lending evidence", "influence": "High", "engagement": "Basel III/IFRS 9 Mapping (Notebook 08)"},
    {"role": "Collections Operations", "interest": "Propensity-to-cure scores for treatment prioritization", "influence": "Medium", "engagement": "Collections Optimization (Problem 9, future phase)"},
    {"role": "Data Engineering", "interest": "Pipeline reliability at 5.5M+ / 11.3M+ row scale", "influence": "Medium", "engagement": "Data Engineering (Notebook 02) design review"},
    {"role": "Card Issuing / Product Business Unit", "interest": "Credit line management and profitability", "influence": "Medium", "engagement": "Risk-Adjusted Profitability Modeling (Problem 13, future phase)"},
    {"role": "MLOps / Platform Engineering", "interest": "Deployment reliability, drift monitoring", "influence": "Medium", "engagement": "FastAPI/Docker/MLOps/Monitoring (Notebooks 09-12)"},
    {"role": "External Regulator (e.g. Federal Reserve / OCC)", "interest": "SR 11-7 model governance evidence", "influence": "High (indirect)", "engagement": "Full documentation trail across Notebooks 01, 07, 08"},
]

stakeholder_csv_path = PILLAR_DIRS["business_documents"] / "Stakeholder_Analysis.csv"
with open(stakeholder_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["role", "interest", "influence", "engagement"])
    writer.writeheader()
    writer.writerows(STAKEHOLDERS)

for s in STAKEHOLDERS:
    print(f"- {s['role']} [{s['influence']} influence] -- {s['interest']}")

print(f"\n\u2705 Saved {len(STAKEHOLDERS)} stakeholders -> {stakeholder_csv_path}")


# =============================================================================
# SECTION 7: RISK APPETITE STATEMENT
# =============================================================================
_section("SECTION 7: Risk Appetite Statement")

RISK_APPETITE = {
    "model_deployment_floor": (
        "A PD model must not be promoted to production scoring with an AMEX competition "
        "metric below 0.75 on a held-out validation set. Notebook 05 checks its own result "
        "against this floor when it runs and reports pass/fail in its own metrics artifact -- "
        "this notebook does not pre-judge that outcome."
    ),
    "monitoring_trigger": (
        "Population Stability Index (PSI) on the scored population exceeding 0.25 against "
        "the training-time distribution triggers mandatory model review (built out in Notebook 12)."
    ),
    "fairness_threshold": (
        "Disparate-impact ratio across any tested proxy segment must remain above 0.80 (the "
        "standard four-fifths rule reference point) before deployment (tested in Notebook 07)."
    ),
    "default_rate_tolerance": (
        "This statement does not itself set portfolio-wide default-rate limits -- that is a "
        "business/board decision outside this technical platform's scope. What this platform "
        "commits to is flagging deviation: any 5-percentage-point swing in observed default "
        f"rate versus the {default_rate:.2%} baseline (computed live in Section 4 from "
        f"train_labels.csv, this run) triggers an alert (Notebook 12)."
    ),
}

for key, value in RISK_APPETITE.items():
    label = key.replace("_", " ").title()
    print(f"\n{label}:\n{value}")


# =============================================================================
# SECTION 8: SMART OBJECTIVES
# =============================================================================
_section("SECTION 8: SMART Objectives")

SMART_OBJECTIVES = [
    {
        "objective": "Baseline PD model (Notebook 05, upcoming)",
        "specific": "Train a PD model on the full, real AMEX training population and validate on a held-out split.",
        "measurable": "AMEX competition metric and AUC on the validation set.",
        "achievable": f"Raw data confirmed present and readable end-to-end this run: {train_customer_count:,} training customers, {test_customer_count:,} test customers.",
        "relevant": "Foundation for all 13 downstream problems in the roadmap.",
        "time_bound": "Complete within Notebook 05's single run.",
        "result": "PENDING -- Notebook 05. This notebook (01) does not train or report model metrics.",
    },
    {
        "objective": "Data Engineering rebuild on Polars (Notebook 02, next)",
        "specific": "Re-implement the train/test aggregation pipeline in Polars with lazy/streaming execution.",
        "measurable": "Wall-clock time for full train_data.csv + test_data.csv aggregation, and peak memory used.",
        "achievable": "Polars' streaming engine is designed exactly for larger-than-RAM columnar aggregation.",
        "relevant": "Directly addresses OOM/timeout risk when processing files of this size.",
        "time_bound": "Complete within Notebook 02's single run.",
        "result": "PENDING -- Notebook 02.",
    },
    {
        "objective": "Full feature set model comparison (Notebook 05, upcoming)",
        "specific": "Train and compare Logistic Regression, Decision Tree, Random Forest, XGBoost, LightGBM, CatBoost, and a Neural Network on the richer engineered feature set.",
        "measurable": "AMEX metric, AUC, and calibration for each of the 7 candidates.",
        "achievable": "Notebook 02's feature store plus this machine's confirmed hardware make a 7-model comparison practical in a single run.",
        "relevant": "Selects the champion model that every later notebook (06-18) builds on.",
        "time_bound": "Complete within Notebook 05's single run.",
        "result": "PENDING -- Notebook 05.",
    },
]

for obj in SMART_OBJECTIVES:
    print(f"\nObjective: {obj['objective']}")
    print(f"  Specific     : {obj['specific']}")
    print(f"  Measurable   : {obj['measurable']}")
    print(f"  Achievable   : {obj['achievable']}")
    print(f"  Relevant     : {obj['relevant']}")
    print(f"  Time-bound   : {obj['time_bound']}")
    print(f"  Result       : {obj['result']}")


# =============================================================================
# SECTION 9: GENERATE DELIVERABLES -- BUSINESS REQUIREMENT DOCUMENT & PROJECT CHARTER
# =============================================================================
_section("SECTION 9: Generate Deliverables")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_kv_table(doc, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for k, v in data.items():
        row = table.add_row().cells
        row[0].text = k.replace("_", " ").title()
        row[1].text = str(v)
    return table


# ---- Business Requirement Document ----
brd = Document()
brd.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
brd.add_paragraph("Business Requirement Document -- Phase 1, Problem 1: Credit Scoring / PD Prediction")
brd.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

_add_heading(brd, "1. Problem Statement", level=1)
brd.add_paragraph(BUSINESS_PROBLEM["problem_statement"])

_add_heading(brd, "2. Why It Matters", level=1)
brd.add_paragraph(BUSINESS_PROBLEM["why_it_matters"])

_add_heading(brd, "3. Success Metric", level=1)
brd.add_paragraph(BUSINESS_PROBLEM["success_metric"])

_add_heading(brd, "4. Current State", level=1)
brd.add_paragraph(BUSINESS_PROBLEM["current_state"])

_add_heading(brd, "5. Dataset Limitation", level=1)
brd.add_paragraph(BUSINESS_PROBLEM["important_dataset_limitation"])

_add_heading(brd, "6. KPI Tree", level=1)
kpi_table = brd.add_table(rows=1, cols=4)
kpi_table.style = "Light Grid Accent 1"
hdr = kpi_table.rows[0].cells
hdr[0].text, hdr[1].text, hdr[2].text, hdr[3].text = "KPI", "Definition", "Current Value", "Computable Now"
for row in KPI_TREE:
    cells_ = kpi_table.add_row().cells
    cells_[0].text = row["kpi"]
    cells_[1].text = row["definition"]
    cells_[2].text = row["current_value"]
    cells_[3].text = str(row["computable_now"])

_add_heading(brd, "7. Risk Appetite Statement", level=1)
for key, value in RISK_APPETITE.items():
    brd.add_paragraph(value, style="List Bullet")

_add_heading(brd, "8. Stakeholders", level=1)
for s in STAKEHOLDERS:
    brd.add_paragraph(f"{s['role']} ({s['influence']} influence): {s['interest']}", style="List Bullet")

brd_path = PILLAR_DIRS["business_documents"] / "Business_Requirement_Document.docx"
brd.save(brd_path)
logger.info(f"Saved {brd_path}")

# ---- Project Charter ----
charter = Document()
charter.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
charter.add_paragraph("Project Charter -- Phase 1, Problem 1: Credit Scoring / PD Prediction")
charter.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

_add_heading(charter, "Project Objective", level=1)
charter.add_paragraph(BUSINESS_PROBLEM["problem_statement"])

_add_heading(charter, "SMART Objectives", level=1)
for obj in SMART_OBJECTIVES:
    charter.add_heading(obj["objective"], level=2)
    _add_kv_table(charter, {
        "Specific": obj["specific"], "Measurable": obj["measurable"],
        "Achievable": obj["achievable"], "Relevant": obj["relevant"],
        "Time-bound": obj["time_bound"], "Result": obj["result"],
    })

_add_heading(charter, "Governance", level=1)
charter.add_paragraph(
    "This project follows CRISP-DM inside Agile sprints, applies the WARP enterprise "
    "code-optimization standard to every notebook, and is grounded in SR 11-7, Basel III, "
    "and IFRS 9 compliance frameworks -- see the Master Execution Plan for full detail."
)

charter_path = PILLAR_DIRS["business_documents"] / "Project_Charter.docx"
charter.save(charter_path)
logger.info(f"Saved {charter_path}")

print(f"\u2705 Business_Requirement_Document.docx -> {brd_path}")
print(f"\u2705 Project_Charter.docx -> {charter_path}")


# =============================================================================
# SECTION 9B: VERIFY DELIVERABLES
# =============================================================================
_section("SECTION 9B: Verify Deliverables Were Written Correctly")

_expected_files = [stakeholder_csv_path, brd_path, charter_path, config_path]

all_ok = True
for fp in _expected_files:
    if fp.exists() and fp.stat().st_size > 0:
        print(f"\u2705 {fp.name:<40} {fp.stat().st_size:>8,} bytes")
    else:
        all_ok = False
        print(f"\u274c MISSING OR EMPTY: {fp}")

if not all_ok:
    raise RuntimeError("One or more Notebook 01 deliverables failed to write. See \u274c lines above.")

print("\nAll Notebook 01 deliverables verified present and non-empty.")


# =============================================================================
# SECTION 10: SMART RECOMMENDATIONS (feeds Notebook 17's rollup)
# =============================================================================
_section("SECTION 10: SMART Recommendations")

notebook_01_recommendations = [
    "Specific: Proceed directly to Notebook 02 (Data Engineering, Polars rewrite) using the "
    "PROJECT_CONFIG just written -- no path or seed redefinition needed.",
    "Measurable: Notebook 02 should report its own wall-clock time and peak memory for the "
    "full train_data.csv (16.4GB) + test_data.csv (33.8GB) aggregation -- these become the "
    "first real, notebook-generated performance numbers in this platform, written to its own "
    "artifact file for Notebook 17 to read.",
    "Achievable: this machine's logical core count and total RAM are detected live in Section "
    "3 above and written to project_config.json for Notebook 02 to read.",
    "Relevant: every one of the 14 roadmap problems depends on Notebook 02's feature store.",
    "Time-bound: target Notebook 02 completion before starting Notebook 03 (Data Validation), "
    "per the Sprint 1 definition of done in the Master Execution Plan.",
]

for i, rec in enumerate(notebook_01_recommendations, 1):
    print(f"{i}. {rec}")

smart_log_path = ARTIFACTS_DIR / "notebook_01_smart_recommendations.json"
with open(smart_log_path, "w", encoding="utf-8") as f:
    json.dump({"notebook": "01_business_understanding", "recommendations": notebook_01_recommendations}, f, indent=2)

print(f"\n\u2705 Saved -> {smart_log_path} (Notebook 17 reads this file to build the rolled-up SMART section)")


# =============================================================================
# SECTION 11: NOTEBOOK 01 COMPLETION SUMMARY
# =============================================================================
_section("SECTION 11: Notebook 01 Complete -- Handoff to Notebook 02")

print("NOTEBOOK 01: BUSINESS UNDERSTANDING -- COMPLETE")
print(f"  Phase                 : {BUSINESS_PROBLEM['phase']}")
print(f"  Problem               : {BUSINESS_PROBLEM['problem_name']}")
print(f"  Files produced        : 4")
print(f"    - {stakeholder_csv_path.name}")
print(f"    - {brd_path.name}")
print(f"    - {charter_path.name}")
print(f"    - project_config.json (shared config for Notebooks 02-18)")
print(f"  Next notebook         : 02_data_engineering.ipynb (Polars rebuild, Sprint 1)")
print("\n\u2705 Ready to proceed.")
